In [26]:
# Cell 0 — Project Imports

import torch

In [27]:
# Cell 1 — Sliding-Window 시작 좌표 계산

def compute_sliding_window_starts(
    image_size: int,
    patch_size: int,
    step_size: int,
) -> list[int]:
    """한 spatial axis 전체를 덮는 patch 시작 좌표 계산"""
    
    # 잘못된 patch or step 크기 차단
    if patch_size <= 0 or step_size <= 0:
        raise ValueError(
            "patch_size & step_size는 양수 필요"
        )
        
    # Patch가 image보다 큰 경우 padding을 전제로 시작점 0 반환
    if patch_size >= image_size:
        return [0]

    maximum_start = image_size - patch_size
    
    start_positions = list(
        range(
            0,
            maximum_start + 1,
            step_size,
        )
    )
    
    # 마지막 patch가 image 끝에 정확히 닿도록 마지막 좌표 추가
    # -> spatial axis 전체 coverage 보장
    if start_positions[-1] != maximum_start:
        start_positions.append(
            maximum_start
        )

    return start_positions


# Volume spatial shape: (D, H, W)
volume_size_zyx: tuple[int, int, int] = (
    20,
    32,
    40,
)
# patch size
sliding_patch_size_zyx: tuple[int, int, int] = (
    8,
    16,
    16,
)
# step size
step_size_zyx: tuple[int, int, int] = (
    4,
    8,
    8,
)

# Depth axis의 patch 시작 좌표 계산
z_starts = compute_sliding_window_starts(
    image_size=volume_size_zyx[0],
    patch_size=sliding_patch_size_zyx[0],
    step_size=step_size_zyx[0],
)

# Height axis의 patch 시작 좌표 계산
y_starts = compute_sliding_window_starts(
    image_size=volume_size_zyx[1],
    patch_size=sliding_patch_size_zyx[1],
    step_size=step_size_zyx[1],
)

# Width axis의 patch 시작 좌표 계산
x_starts = compute_sliding_window_starts(
    image_size=volume_size_zyx[2],
    patch_size=sliding_patch_size_zyx[2],
    step_size=step_size_zyx[2],
)

# 세 axis 시작 좌표의 모든 조합이 전체 3D patch 개수
number_of_patches = (
    len(z_starts)
    * len(y_starts)
    * len(x_starts)
)

print("Volume size:", volume_size_zyx)
print("Patch size: ", sliding_patch_size_zyx)
print("Step size:  ", step_size_zyx)
print()

print("Z starts:", z_starts)
print("Y starts:", y_starts)
print("X starts:", x_starts)
print()

print("Number of Z positions:", len(z_starts))
print("Number of Y positions:", len(y_starts))
print("Number of X positions:", len(x_starts))
print("Total patch count:    ", number_of_patches)

Volume size: (20, 32, 40)
Patch size:  (8, 16, 16)
Step size:   (4, 8, 8)

Z starts: [0, 4, 8, 12]
Y starts: [0, 8, 16]
X starts: [0, 8, 16, 24]

Number of Z positions: 4
Number of Y positions: 3
Number of X positions: 4
Total patch count:     48


In [28]:
# Cell 2 — 겹치는 3D Patch 추출

def extract_sliding_window_patches(
    volume: torch.Tensor,                       # [B, C, D, H, W]
    patch_size_zyx: tuple[int, int, int],       # (patch_D, patch_H, patch_W)
    step_size_zyx: tuple[int, int, int],        # (step_D, step_H, step_W)
) -> tuple[
    list[torch.Tensor],                         # 각 원소: [B, C, patch_D, patch_H, patch_W]
    list[tuple[int, int, int]],                 # 각 원소: (start_z, start_y, start_x)
]:
    """Volume 전체를 덮는 overlapping 3D patch와 시작 좌표 추출"""

    # Input volume의 spatial 크기 분리: 
    # Volume: (D, H, W)
    volume_depth, volume_height, volume_width = (
        volume.shape[-3:]
    )

    # Patch의 spatial 크기 분리
    # Patch size: (patch_D, patch_H, patch_W)
    patch_depth, patch_height, patch_width = (
        patch_size_zyx
    )

    # 각 spatial axis의 patch 시작 좌표 계산
    z_positions = compute_sliding_window_starts(
        image_size=volume_depth,
        patch_size=patch_depth,
        step_size=step_size_zyx[0],
    )
    y_positions = compute_sliding_window_starts(
        image_size=volume_height,
        patch_size=patch_height,
        step_size=step_size_zyx[1],
    )
    x_positions = compute_sliding_window_starts(
        image_size=volume_width,
        patch_size=patch_width,
        step_size=step_size_zyx[2],
    )


    sliding_patches: list[torch.Tensor] = []
    patch_origins_zyx: list[
        tuple[int, int, int]
    ] = []

    # Z, Y, X 시작 좌표의 모든 조합 순회
    for start_z in z_positions:
        for start_y in y_positions:
            for start_x in x_positions:

                # 현재 시작 좌표에 해당하는 실제 3D patch 추출
                patch = volume[
                    :,
                    :,
                    start_z:start_z + patch_depth,
                    start_y:start_y + patch_height,
                    start_x:start_x + patch_width,
                ]  # [B, C, patch_D, patch_H, patch_W]

                sliding_patches.append(patch) # 1) 추출된 patch 저장
                patch_origins_zyx.append(     # 2) volume 내부 시작 좌표 저장
                    (
                        start_z,
                        start_y,
                        start_x,
                    )
                )

    return sliding_patches, patch_origins_zyx


# 위치 확인용 synthetic volume 생성
# 각 voxel 값을 고유한 정수로 설정
synthetic_volume = torch.arange(
    1 * 1 * 20 * 32 * 40,
    dtype=torch.float32,
).reshape(
    1,
    1,
    20,
    32,
    40,
)  # [B=1, C=1, D=20, H=32, W=40]

# Volume 전체에서 overlapping patch 추출
sliding_patches, patch_origins_zyx = (
    extract_sliding_window_patches(
        volume=synthetic_volume,
        patch_size_zyx=sliding_patch_size_zyx,
        step_size_zyx=step_size_zyx,
    )
)

# X축 방향으로 이웃한 첫 번째와 두 번째 patch 선택
first_patch = sliding_patches[0]
second_patch = sliding_patches[1]

first_origin = patch_origins_zyx[0]
second_origin = patch_origins_zyx[1]

# 두 patch가 공유하는 동일한 global voxel 값 비교
first_patch_overlap_value = first_patch[
    :,
    :,
    0,
    0,
    8,
].item()

second_patch_overlap_value = second_patch[
    :,
    :,
    0,
    0,
    0,
].item()

print("Volume shape:", synthetic_volume.shape)
print("Patch count:", len(sliding_patches))
print("Patch shape:", first_patch.shape)
print()

print("First origin: ", first_origin)
print("Second origin:", second_origin)
print()

print(
    "First patch overlap value: ",
    first_patch_overlap_value,
)
print(
    "Second patch overlap value:",
    second_patch_overlap_value,
)
print(
    "Same global voxel:",
    first_patch_overlap_value
    == second_patch_overlap_value,
)

Volume shape: torch.Size([1, 1, 20, 32, 40])
Patch count: 48
Patch shape: torch.Size([1, 1, 8, 16, 16])

First origin:  (0, 0, 0)
Second origin: (0, 0, 8)

First patch overlap value:  8.0
Second patch overlap value: 8.0
Same global voxel: True


In [29]:
# Cell 3 — Patch Prediction 누적과 Count Map

# accumulated_predictions
# = 겹치는 prediction을 전부 더한 값

# overlap_count_map
# = 몇 번 겹쳤는지

# 최종 prediction
# = accumulated_predictions / overlap_count_map

def accumulate_patch_predictions(
    patch_predictions: list[torch.Tensor],         # list: (B, K, patch_D, patch_H, patch_W)
    patch_origins_zyx: list[tuple[int, int, int]], # list: (start_z, start_y, start_x)
    output_shape: tuple[int, int, int, int, int],  # (B, K, D, H, W)
) -> tuple[
    torch.Tensor,  # [B, K, D, H, W]
    torch.Tensor,  # [1, 1, D, H, W]
]:
    """Patch prediction 누적과 voxel별 overlap 횟수 계산"""
    
    # 비어 있는 prediction 입력 차단
    if len(patch_predictions) == 0:
        raise ValueError(
            "patch_predictions에 하나 이상의 Tensor 필요"
        )

    # Prediction과 좌표 개수 불일치 차단
    if len(patch_predictions) != len(
        patch_origins_zyx
    ):
        raise ValueError(
            "patch prediction과 origin 개수 일치 필요"
        )

    first_prediction = patch_predictions[0]

    # 모든 patch prediction을 합산할 전체 volume 생성
    accumulated_predictions = torch.zeros(
        output_shape,
        dtype=first_prediction.dtype,
        device=first_prediction.device,
    )  # [B, K, D, H, W]

    # 각 voxel에 prediction이 더해진 횟수 기록
    # Batch와 class에 동일하게 broadcast 가능한 shape 사용
    overlap_count_map = torch.zeros(
        (
            1,
            1,
            output_shape[2],
            output_shape[3],
            output_shape[4],
        ),
        dtype=first_prediction.dtype,
        device=first_prediction.device,
    )  # [1, 1, D, H, W]

    # 각 prediction과 원래 volume 내부 위치를 함께 순회
    for patch_prediction, patch_origin in zip(
        patch_predictions,
        patch_origins_zyx,
    ):
        start_z, start_y, start_x = patch_origin

        patch_depth, patch_height, patch_width = (
            patch_prediction.shape[-3:]
        )

        end_z = start_z + patch_depth
        end_y = start_y + patch_height
        end_x = start_x + patch_width

        # Patch prediction을 대응하는 global 영역에 누적
        accumulated_predictions[
            :,
            :,
            start_z:end_z,
            start_y:end_y,
            start_x:end_x,
        ] += patch_prediction

        # 동일한 global 영역의 overlap 횟수 1 증가
        overlap_count_map[
            :,
            :,
            start_z:end_z,
            start_y:end_y,
            start_x:end_x,
        ] += 1

    return (
        accumulated_predictions,
        overlap_count_map,
    )


# Synthetic patch를 model prediction의 대역으로 사용
# clone으로 원본 volume view와 별도 Tensor 생성
synthetic_patch_predictions = [
    patch.clone()
    for patch in sliding_patches
]

prediction_volume_shape: tuple[
    int,
    int,
    int,
    int,
    int,
] = (
    synthetic_volume.shape[0],
    synthetic_volume.shape[1],
    synthetic_volume.shape[2],
    synthetic_volume.shape[3],
    synthetic_volume.shape[4],
)

# Patch prediction 합산과 voxel별 overlap 횟수 계산
accumulated_predictions, overlap_count_map = (
    accumulate_patch_predictions(
        patch_predictions=synthetic_patch_predictions,
        patch_origins_zyx=patch_origins_zyx,
        output_shape=prediction_volume_shape,
    )
)

# 여러 patch가 겹치는 volume 내부 voxel 선택
inspection_index = (
    0,
    0,
    10,
    16,
    20,
)

original_value = synthetic_volume[
    inspection_index
].item()

accumulated_value = accumulated_predictions[
    inspection_index
].item()

overlap_count = overlap_count_map[
    0,
    0,
    inspection_index[2],
    inspection_index[3],
    inspection_index[4],
].item()

print(
    "Accumulated prediction shape:",
    accumulated_predictions.shape,
)
print(
    "Count map shape:              ",
    overlap_count_map.shape,
)
print()

print(
    "Minimum overlap count:",
    overlap_count_map.min().item(),
)
print(
    "Maximum overlap count:",
    overlap_count_map.max().item(),
)
print()

print("Original voxel value:   ", original_value)
print("Accumulated voxel value:", accumulated_value)
print("Voxel overlap count:    ", overlap_count)
print(
    "Original × count:       ",
    original_value * overlap_count,
)

Accumulated prediction shape: torch.Size([1, 1, 20, 32, 40])
Count map shape:               torch.Size([1, 1, 20, 32, 40])

Minimum overlap count: 1.0
Maximum overlap count: 8.0

Original voxel value:    13460.0
Accumulated voxel value: 107680.0
Voxel overlap count:     8.0
Original × count:        107680.0


In [30]:
# Cell 4 — Overlap Normalization과 Volume 복원

def normalize_overlapping_predictions(
    accumulated_predictions: torch.Tensor, # [B, K, D, H, W]
    overlap_count_map: torch.Tensor,       # [1, 1, D, H, W]
) -> torch.Tensor:
    """누적된 prediction을 voxel별 overlap 횟수로 정규화"""

    # Sliding window가 덮지 못한 voxel 존재 여부 검사
    if torch.any(overlap_count_map == 0):
        raise ValueError(
            "count가 0인 미복원 voxel 존재"
        )

    # 겹친 patch prediction의 voxel별 평균 계산
    normalized_predictions = (
        accumulated_predictions
        / overlap_count_map
    )  # [B, K, D, H, W]

    return normalized_predictions



# 겹치는 영역을 단순 합산하지 않고 해당 voxel이 등장한 횟수로 나눠 평균 prediction을 복원
reconstructed_volume = (
    normalize_overlapping_predictions(
        accumulated_predictions=accumulated_predictions,
        overlap_count_map=overlap_count_map,
    )
)  # [B=1, K=1, D=20, H=32, W=40]

# 원본과 복원 volume의 voxel별 절대 오차 계산
absolute_error = torch.abs(
    reconstructed_volume
    - synthetic_volume
)  # [B=1, K=1, D=20, H=32, W=40]

maximum_absolute_error = (
    absolute_error.max().item()
)

mean_absolute_error = (
    absolute_error.mean().item()
)

# 모든 voxel의 sliding-window coverage 확인
all_voxels_covered = torch.all(
    overlap_count_map > 0
).item()

# 복원 volume과 원본 volume의 전체 일치 확인
volume_reconstructed_correctly = torch.allclose(
    reconstructed_volume,
    synthetic_volume,
)

# 이전 Cell 3에서 검사한 내부 voxel의 복원값 확인
reconstructed_inspection_value = (
    reconstructed_volume[
        inspection_index
    ].item()
)

print(
    "Reconstructed shape:",
    reconstructed_volume.shape,
)
print(
    "All voxels covered: ",
    all_voxels_covered,
)
print(
    "Volume reconstructed:",
    volume_reconstructed_correctly,
)
print()

print(
    "Original inspection value:     ",
    original_value,
)
print(
    "Reconstructed inspection value:",
    reconstructed_inspection_value,
)
print()

print(
    "Maximum absolute error:",
    maximum_absolute_error,
)
print(
    "Mean absolute error:   ",
    mean_absolute_error,
)

Reconstructed shape: torch.Size([1, 1, 20, 32, 40])
All voxels covered:  True
Volume reconstructed: True

Original inspection value:      13460.0
Reconstructed inspection value: 13460.0

Maximum absolute error: 0.0
Mean absolute error:    0.0
